# The halting problem with a batch of Turing machines

*others · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/turing_halting)

Not physics, just for fun: run hundreds of thousands of Turing machines at once as tensors. One step of every machine is a gather (read the tape under each head), an advanced-indexing look-up of its rule and a scatter (write). Enumerate all 2- and 3-state busy beaver candidates, find the champions $S(2) = 6$ and $S(3) = 21$, and watch the halting problem in action: whatever step budget you pick, some machines are still undecided.

**Vector form**

$$\delta(q, r) = (w, m, q'),\qquad S(n) = \max_{M \text{ halts}} \tau(M),\qquad f(t) = \frac{\#\{M : \tau(M) \le t\}}{\#\{M\}}$$

**Index (tensor) form**

$$r_b = T_{b,h_b},\qquad (w_b, m_b, q'_b) = \delta_{b,\,s_b,\,r_b},\qquad T_{b,h_b} \leftarrow w_b,\quad h_b \leftarrow h_b + m_b,\quad s_b \leftarrow q'_b \quad (s_b \ne H)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\delta_{bqr}$ | `all_machines(n, normalise, device)` | (B, n, 2, 3) | rule table of machine $b$ for state $q$ and read symbol $r$: (write $w$, move $m = \pm1$, next state $q'$); state $n$ is the halt state $H$ |
| $T_{bx}$ | `blank(B, L, device)` | (B, L) | tape cell $x$ of machine $b$, all 0 at the start; $L = 2t_{\max} + 1$ so no head can reach the end |
| $h_b,\ s_b$ | `head, state` | (B,) | head position (starting in the middle) and current state (starting in $A$) |
| $r_b = T_{b,h_b}$ | `tape.gather(1, head[:, None])` | (B,) | read the symbol under every head at once |
| $\delta_{b,s_b,r_b}$ | `table[arange(B), state, r]` | (B, 3) | advanced indexing: each machine looks up its own rule |
| $T_{b,h_b} \leftarrow w_b$ | `tape.scatter_(1, head[:, None], w[:, None])` | (B, L) | write; then move and change state, only where $s_b \ne H$ (halted machines freeze) |
| $\tau_b$ | `run(table, budget)['halt_time']` | (B,) | step on which machine $b$ entered $H$, or $-1$: undecided within the budget |
| $f(t)$ | `halted_fraction(halt_time, budget)` | (t_max+1,) | fraction of machines with $\tau_b \le t$: non-decreasing, and stuck below 1 |
| $\Sigma_b = \sum_x T_{bx}$ | `tape.sum(dim=1)` | (B,) | repeated $x$ ⇒ summed: the number of 1s a halted machine leaves |

Turing (1936) showed that no program can decide, for every machine and input, whether it halts. The diagonal argument in one paragraph: suppose a machine $D(M)$ answered "halts" or "runs forever" for every machine $M$ started on its own description. Build $G$ that runs $D(M)$ and then does the opposite: loop forever if $D$ says "halts", halt if $D$ says "runs forever". Now ask what $G$ does on its own description: if it halts, $D$ said it runs forever, and if it runs forever, $D$ said it halts. Either way $D$ is wrong, so $D$ cannot exist. Radó (1962) turned this into a game, the busy beaver: among all $n$-state, 2-symbol machines started on a blank tape that do halt, $S(n)$ is the longest run and $\Sigma(n)$ the most 1s left behind. Both grow faster than any computable function, since a program for $S(n)$ would decide halting: run each machine $S(n)$ steps, and if it has not halted by then it never will. Known values: $S(2) = 6$, $\Sigma(2) = 4$ (Radó), $S(3) = 21$, $\Sigma(3) = 6$ (Lin and Radó 1965), $S(4) = 107$, $\Sigma(4) = 13$ (Brady 1983), and $S(5) = 47\,176\,870$, proved in 2024 by the bbchallenge collaboration with a Coq-verified proof that every other 5-state machine runs forever. The plot shows why this is hard: the halted fraction $f(t)$ rises and then flattens, and a finite budget can never tell a slow halter from a machine that runs forever. Here the plateau after step 21 is the known answer for 3 states, because the machines left over were proved to loop. Normalisation for 3 states: the first rule is fixed to A0 → 1RB, a common choice that keeps the known champions (by left–right symmetry the first move can be to the right, the names of the other states are interchangeable, and a machine that stays in A on a blank tape never halts), and every halting rule is 1RH, giving $13^5 = 371\,293$ machines instead of $16^6$. Try 4 states: a random sample rarely contains the 107-step champion.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/turing_halting/main.py
"""Main loop: enumerate machines → run them all at once → find the busy beavers → plot."""

import time
from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors

KNOWN = {2: (6, 4), 3: (21, 6), 4: (107, 13)}  # (S(n), Σ(n)): Radó 1962, Lin & Radó 1965, Brady 1983


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, budget = params["states"], params["budget"]
    tic = time.perf_counter()

    # 1. Every 2-state machine, no shortcuts: (2·2·3)^4 = 20 736 transition tables in one tensor.
    two = tensors.all_machines(2, normalise=False, device=dev)  # (B, 2, 2, 3)
    res2 = solve.run(two, budget)
    halted2 = res2["halt_time"] >= 0
    s2, sigma2 = int(res2["halt_time"].max()), int(res2["ones"][halted2].max())
    print(f"2 states: all {len(two):,} machines, budget {budget} steps: {int(halted2.sum()):,} halt, "
          f"S(2) = {s2} {'✓' if s2 == 6 else '✗'}, Σ(2) = {sigma2} {'✓' if sigma2 == 4 else '✗'}")  # fmt: skip

    # 2. The chosen number of states: exhaustive (normalised) for n ≤ 3, a random sample for n = 4.
    if n == 2:
        table, res, what = two, res2, "all 2-state machines"
    else:
        if n == 3:
            table, what = tensors.all_machines(3, normalise=True, device=dev), "all 3-state machines (A0 = 1RB)"
        else:
            table = tensors.random_machines(4, params["n_sample"], seed=0, device=dev)
            what = f"{params['n_sample']:,} random 4-state machines"
        res = solve.run(table, budget)
    print(f"tape {2 * budget + 1} cells × {len(table):,} machines = {res['tape'].numel() / 1e6:.0f} M cells")
    halt_time, ones = res["halt_time"], res["ones"]
    halted = halt_time >= 0
    s_n, sigma_n = int(halt_time.max()), int(ones[halted].max())
    frac = solve.halted_fraction(halt_time, budget)  # (budget + 1,)
    exact = n <= 3
    s_known, sigma_known = KNOWN[n]
    mark = (lambda found, known: "✓" if found == known else "✗") if exact else (lambda found, known: "")  # noqa: E731
    print(f"{what}: {int(halted.sum()):,} halt within {budget} steps, {int((~halted).sum()):,} undecided "
          f"({int(res['off_tape'].sum())} ran off the tape)")  # fmt: skip
    print(f"  longest run: S = {s_n} steps {mark(s_n, s_known)}   most 1s: Σ = {sigma_n} {mark(sigma_n, sigma_known)}"
          f"   (known S({n}) = {s_known}, Σ({n}) = {sigma_known})")  # fmt: skip
    best = solve.champions(halt_time, ones)
    print(f"  step champion {tensors.to_text(table[best[s_n]])},"
          f"  ones champion {tensors.to_text(table[int((ones * halted).argmax())])}")  # fmt: skip
    for t in sorted({1, 2, 5, 10, 20, 50, 100, 200, 500, budget} & set(range(budget + 1))):
        print(f"  budget {t:4d}: halted {frac[t]:7.2%}   undecided {1 - frac[t]:7.2%}")
    last = int(halt_time.max())
    print(f"  nothing halts after step {last}: the remaining {1 - frac[-1]:.2%} look like they run forever, "
          f"but no budget can prove it in general (the halting problem)")  # fmt: skip

    # 3. The known 4-state champion, run alone.
    bb4 = tensors.parse(tensors.CHAMPIONS["BB(4)"])
    one = solve.run(bb4[None], 200)
    s4, sigma4 = int(one["halt_time"][0]), int(one["ones"][0])
    print(f"BB(4) champion {tensors.CHAMPIONS['BB(4)']}: halts after {s4} steps {'✓' if s4 == 107 else '✗'} "
          f"with {sigma4} ones {'✓' if sigma4 == 13 else '✗'}")  # fmt: skip
    print(f"simulated {len(two) + (len(table) if n > 2 else 0) + 1:,} machines in {time.perf_counter() - tic:.1f} s "
          f"on {dev}")  # fmt: skip

    records = {s: solve.space_time(table[i], s) for s, i in best.items()}  # champion halting at each step count s
    check = f"S({n})={s_n} {mark(s_n, s_known)}  Σ({n})={sigma_n} {mark(sigma_n, sigma_known)}"
    if not exact:
        check = f"best in sample: S={s_n}, Σ={sigma_n}\n(true S(4)=107, Σ(4)=13)"
    ok4 = f"S={s4} {'✓' if s4 == 107 else '✗'}  Σ={sigma4} {'✓' if sigma4 == 13 else '✗'}"
    labels = {"batch": what, "check": check, "bb4": f"BB(4): {ok4}"}
    return plot.animate(records, frac, solve.halted_fraction(res2["halt_time"], budget), solve.space_time(bb4, s4),
                        labels, out_dir / "turing_halting.gif")  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/turing_halting/equation.py
r"""Turing machine → tensor translation.

A Turing machine (Turing 1936) is a head on an infinite tape of 0s and 1s, in one of finitely many states q. Each
step it reads the symbol r under the head and looks up its rule δ(q, r) = (w, m, q'): write w, move m = ±1, go to
state q'. Reaching the halt state H stops it. For a batch of B machines b, each with its own table δ_b, one step is

    r_b = T_{b, h_b}                          (read: gather along the tape)
    (w_b, m_b, q'_b) = δ_{b, s_b, r_b}        (look up the rule: advanced indexing)
    T_{b, h_b} ← w_b,   h_b ← h_b + m_b,   s_b ← q'_b      (write: scatter; move; change state)

applied only where machine b is still running (s_b ≠ H). No index is summed: every machine is independent, so one
step of the whole batch is one gather, one look-up and one scatter, whatever B is.
"""

import torch


def read(tape: torch.Tensor, head: torch.Tensor) -> torch.Tensor:
    r"""r_b = T_{b, h_b}.   tape (B, L), head (B,) → (B,)"""
    return tape.gather(1, head[:, None])[:, 0]


def rule(table: torch.Tensor, state: torch.Tensor, symbol: torch.Tensor) -> torch.Tensor:
    r"""(w, m, q')_b = δ_{b, s_b, r_b}.   table (B, n, 2, 3), state (B,), symbol (B,) → (B, 3)"""
    b = torch.arange(len(table), device=table.device)
    q = state.clamp(max=table.shape[1] - 1)  # a halted machine (s = H = n) looks up any row; it is masked out
    return table[b, q, symbol.long()]


def step(
    tape: torch.Tensor, head: torch.Tensor, state: torch.Tensor, table: torch.Tensor, active: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    r"""One step of every machine where active (B,) is True; the others stay frozen.

    tape (B, L) uint8, head (B,), state (B,), table (B, n, 2, 3) → new (tape, head, state)
    """
    r = read(tape, head)  # (B,)
    w, m, q = rule(table, state, r).unbind(-1)  # each (B,)
    write = torch.where(active, w.to(tape.dtype), r)  # a frozen machine "writes" what is already there
    tape.scatter_(1, head[:, None], write[:, None])  # T_{b, h_b} ← w_b, in place: O(B) work, not O(B·L)
    head = head + torch.where(active, m, 0)  # h_b ← h_b + m_b
    state = torch.where(active, q, state)  # s_b ← q'_b
    return tape, head, state


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/turing_halting/tensors.py
"""Build the concrete tensors: transition tables of whole batches of Turing machines, and blank tapes.

A machine with n states (A, B, C, …) and 2 symbols has a rule for every (state, symbol read) pair: the symbol to
write, the move (−1 left, +1 right) and the next state. States are numbered 0 … n−1 and the halt state H is n.
"""

import torch

# Champions in the bbchallenge notation: per state "write move next" for symbol 0 then 1, H = halt.
CHAMPIONS = {
    "BB(2)": "1RB1LB_1LA1RH",  # S(2) = 6 steps, Σ(2) = 4 ones
    "BB(3) steps": "1RB1RH_1LB0RC_1LC1LA",  # S(3) = 21 steps (leaves 5 ones)
    "BB(3) ones": "1RB1RH_0RC1RB_1LC1LA",  # Σ(3) = 6 ones (in 14 steps)
    "BB(4)": "1RB1LB_1LA0LC_1RH1LD_1RD0RA",  # S(4) = 107 steps, Σ(4) = 13 ones (Brady 1983)
}
LETTERS = "ABCDEFGH"


def parse(text: str) -> torch.Tensor:
    """'1RB1LB_1LA1RH' → transition table δ of shape (n, 2, 3): δ[q, r] = (write, move, next)."""
    states = text.split("_")
    n = len(states)
    rule = lambda s: [int(s[0]), 1 if s[1] == "R" else -1, n if s[2] in "HZ" else LETTERS.index(s[2])]  # noqa: E731
    return torch.tensor([[rule(row[0:3]), rule(row[3:6])] for row in states])


def to_text(table: torch.Tensor) -> str:
    """Inverse of parse, for printing: (n, 2, 3) → '1RB1LB_1LA1RH'."""
    n = table.shape[0]
    cell = lambda w, m, q: f"{w}{'R' if m > 0 else 'L'}{'H' if q == n else LETTERS[q]}"  # noqa: E731
    return "_".join("".join(cell(*table[q, r].tolist()) for r in range(2)) for q in range(n))


def rule_options(n: int, normalise: bool) -> torch.Tensor:
    """Every possible single rule (write, move, next) as rows of a (K, 3) tensor.

    Exhaustive: K = 2 writes × 2 moves × (n + 1) next states. Normalised: a halting rule is always 1RH, because
    what it writes and where it moves changes neither when the machine halts nor (except for one more 1) its score.
    """
    rows = [[w, m, q] for q in range(n) for w in (0, 1) for m in (-1, 1)]
    rows += [[1, 1, n]] if normalise else [[w, m, n] for w in (0, 1) for m in (-1, 1)]
    return torch.tensor(rows)


def all_machines(n: int, normalise: bool, device: torch.device) -> torch.Tensor:
    """Every n-state machine as one batch: (B, n, 2, 3).

    Exhaustive: B = (4(n+1))^{2n}, all 20 736 machines for n = 2. Normalised: the first rule is fixed to
    A0 → 1RB (a machine that starts by writing 1, by left–right symmetry moving right, into a new state; the
    other first moves either halt at once or loop forever) and halting rules are 1RH, so B = (4n+1)^{2n−1},
    371 293 machines for n = 3.
    """
    options = rule_options(n, normalise).to(device)  # (K, 3)
    k, free = len(options), 2 * n - (1 if normalise else 0)
    code = torch.arange(k**free, device=device)  # (B,) machine number written in base K
    digits = torch.stack([(code // k**i) % k for i in range(free)], dim=1)  # (B, free)
    table = options[digits]  # (B, free, 3)
    if normalise:
        first = torch.tensor([1, 1, 1], device=device).expand(len(code), 1, 3)  # A0 → 1RB
        table = torch.cat([first, table], dim=1)
    return table.reshape(-1, n, 2, 3)


def random_machines(n: int, count: int, seed: int, device: torch.device) -> torch.Tensor:
    """A random sample of normalised n-state machines (A0 → 1RB, halting rules 1RH): (count, n, 2, 3)."""
    options = rule_options(n, normalise=True)  # (K, 3)
    gen = torch.Generator().manual_seed(seed)
    digits = torch.randint(len(options), (count, 2 * n - 1), generator=gen)
    first = torch.tensor([1, 1, 1]).expand(count, 1, 3)
    return torch.cat([first, options[digits]], dim=1).reshape(count, n, 2, 3).to(device)


def blank(batch: int, length: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """All-zero tapes T (B, L) uint8, heads h (B,) in the middle cell, states s (B,) = 0 (state A)."""
    tape = torch.zeros(batch, length, dtype=torch.uint8, device=device)
    head = torch.full((batch,), length // 2, dtype=torch.long, device=device)
    state = torch.zeros(batch, dtype=torch.long, device=device)
    return tape, head, state


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/turing_halting/solve.py
"""Run a whole batch of Turing machines for a fixed step budget and record when (and whether) each one halts."""

import torch

from problems._common.progress import steps

from .equation import step
from .tensors import blank


def run(table: torch.Tensor, budget: int, length: int | None = None) -> dict[str, torch.Tensor]:
    """Simulate every machine of table (B, n, 2, 3) for at most `budget` steps on a blank tape.

    The tape has L = 2·budget + 1 cells by default with the head in the middle, so no machine can reach its end;
    with a shorter tape a machine that would step off it is frozen and counted as undecided, like one still running.
    Returns halt_time (B,) (the step on which it entered H, −1 if undecided), ones (B,) (1s left on the tape),
    off_tape (B,) and the final tape (B, L).
    """
    n = table.shape[1]
    length = length or 2 * budget + 1
    tape, head, state = blank(len(table), length, table.device)
    halt_time = torch.full_like(state, -1)
    off = torch.zeros_like(state, dtype=torch.bool)
    for t in steps(budget, "steps"):
        active = (state < n) & ~off  # still running and still on the tape
        if not active.any():
            break
        tape, head, state = step(tape, head, state, table, active)
        halt_time = torch.where(active & (state == n), t + 1, halt_time)  # entered H on step t + 1
        off |= (head < 0) | (head >= length)
        head = head.clamp(0, length - 1)  # an off-tape machine is frozen at the edge
    ones = tape.sum(dim=1, dtype=torch.long)
    return {"halt_time": halt_time, "ones": ones, "off_tape": off & (halt_time < 0), "tape": tape}


def halted_fraction(halt_time: torch.Tensor, budget: int) -> torch.Tensor:
    """f(t) = fraction of machines that halted within t steps, t = 0 … budget: (budget + 1,)."""
    t = torch.arange(budget + 1, device=halt_time.device)
    halted = (halt_time[None, :] >= 0) & (halt_time[None, :] <= t[:, None])  # (budget + 1, B)
    return halted.double().mean(dim=1)


def champions(halt_time: torch.Tensor, ones: torch.Tensor) -> dict[int, int]:
    """For every step count s at which some machine halts: the index of the machine that halts at s with the most
    1s. The champion within a budget t is then the entry with the largest s ≤ t."""
    out = {}
    for s in halt_time[halt_time >= 0].unique().tolist():
        idx = torch.nonzero(halt_time == s)[:, 0]
        out[int(s)] = int(idx[ones[idx].argmax()])
    return out


def space_time(table: torch.Tensor, n_steps: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Tape and head position after every step of one machine (n, 2, 3): rows (n_steps + 1, L), heads (n_steps + 1,).
    The tape is L = 2·n_steps + 1 cells with the head starting in the middle."""
    n, length = table.shape[0], 2 * n_steps + 1
    tape, head, state = blank(1, length, table.device)
    rows, heads = [tape[0].clone()], [int(head[0])]
    for _ in range(n_steps):
        tape, head, state = step(tape, head, state, table[None], state < n)
        rows.append(tape[0].clone())
        heads.append(int(head[0]))
    return torch.stack(rows), torch.tensor(heads)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/turing_halting/plot.py
"""Animate the space-time diagram of the current champion next to the halted fraction against the step budget."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from matplotlib.ticker import MaxNLocator

from problems._common.plotting import frame_indices, save_gif

# 0 blank cell, 1 a one, 2 head on a 0, 3 head on a 1, 4 after the machine halted
CMAP = ListedColormap(["#ffffff", "#2563eb", "#fcd34d", "#b45309", "#f3f4f6"])


def picture(rows: torch.Tensor, heads: torch.Tensor, n_rows: int, lo: int, hi: int) -> np.ndarray:
    """One space-time image (n_rows, hi − lo + 1): row t = the tape after t steps, cells lo … hi around the start."""
    start = rows.shape[1] // 2
    img = np.full((n_rows, hi - lo + 1), 4)
    cols = np.arange(lo, hi + 1) + start
    ok = (cols >= 0) & (cols < rows.shape[1])
    tape = rows.cpu().numpy()
    img[: len(tape), ok] = tape[:, cols[ok]]
    for t, h in enumerate(heads.tolist()):
        if lo <= h - start <= hi:
            img[t, h - start - lo] += 2
    return img


def extent(records: list[tuple[torch.Tensor, torch.Tensor]]) -> tuple[int, int]:
    """Smallest cell range (relative to the start) holding every 1 and head position of the given runs."""
    lo, hi = 0, 0
    for rows, heads in records:
        start = rows.shape[1] // 2
        used = np.nonzero(rows.cpu().numpy().any(axis=0))[0].tolist() + heads.tolist()
        lo, hi = min(lo, min(used) - start), max(hi, max(used) - start)
    return lo - 1, hi + 1


def animate(
    records: dict[int, tuple[torch.Tensor, torch.Tensor]],
    frac: torch.Tensor,
    frac2: torch.Tensor,
    bb4: tuple[torch.Tensor, torch.Tensor],
    labels: dict[str, str],
    path: Path,
) -> Path:
    """records: halting step s → (rows, heads) of the champion halting at s; frac (T+1,) halted fraction of the
    chosen n-state batch, frac2 the same for all 2-state machines; bb4 the space-time run of the BB(4) champion."""
    budget = len(frac) - 1
    s_max = max(records)
    lo, hi = extent(list(records.values()))
    images = {s: picture(r, h, s_max + 1, lo, hi) for s, (r, h) in records.items()}
    blank = np.full((s_max + 1, hi - lo + 1), 4)
    lo4, hi4 = extent([bb4])
    bb4_img = picture(*bb4, len(bb4[0]), lo4, hi4)

    sweep = np.unique(np.round(np.geomspace(1, budget, 90)).astype(int))  # budgets t, log-spaced
    sweep = sweep[frame_indices(len(sweep), 84)]
    tail = np.linspace(1, len(bb4[0]), 30).round().astype(int)  # then the BB(4) champion row by row
    frames = [("sweep", int(t)) for t in sweep] + [("bb4", int(k)) for k in tail]
    frames = [frames[i] for i in frame_indices(len(frames), 120)]

    fig, (ax_d, ax_f) = plt.subplots(1, 2, figsize=(9, 4.2), gridspec_kw={"width_ratios": [1, 1.25]})
    image = ax_d.imshow(blank, cmap=CMAP, vmin=-0.5, vmax=4.5, interpolation="nearest", aspect="auto")
    ax_d.set_xticks([])
    ax_d.set_xlabel("tape cells (blue = 1, amber = head)")
    ax_d.set_ylabel("step (time runs down)")
    ax_d.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax_d.grid(False)
    title = ax_d.set_title("champion within 100 steps: 21 steps", fontsize=10)  # placeholder sized for the layout

    t_axis = np.arange(budget + 1)
    f, f2 = frac.cpu().numpy(), frac2.cpu().numpy()
    ax_f.plot(t_axis[1:], f2[1:], color="#9ca3af", lw=1.2, label="all 2-state machines")
    ax_f.plot(t_axis[1:], f[1:], color="#2563eb", lw=1.8, label=labels["batch"])
    shade = [ax_f.fill_between(t_axis[1:2], f[1:2], 1, color="#dc2626", alpha=0.12)]
    cursor = ax_f.axvline(1, color="#f59e0b", lw=1.2)
    ax_f.set_xscale("log")
    ax_f.set_xlim(1, budget)
    ax_f.set_ylim(0, 1.02)
    ax_f.set_xlabel("step budget $t$")
    ax_f.set_ylabel("fraction halted within $t$ steps")
    ax_f.legend(loc="lower right", fontsize=8)
    status = ax_f.text(0.03, 0.97, "", transform=ax_f.transAxes, va="top", fontsize=9, family="monospace",
                       bbox={"facecolor": "white", "edgecolor": "#e5e7eb", "alpha": 0.9})  # fmt: skip

    def update(frame: tuple[str, int]):
        kind, t = frame
        if kind == "sweep":
            done = [s for s in records if s <= t]
            image.set_data(images[max(done)] if done else blank)
            image.set_extent((-0.5, hi - lo + 0.5, s_max + 0.5, -0.5))
            title.set_text(f"champion within {t} steps: {max(done) if done else '-'} steps")
            shade[0].remove()
            shade[0] = ax_f.fill_between(t_axis[1 : t + 1], f[1 : t + 1], 1, color="#dc2626", alpha=0.12, lw=0)
            cursor.set_xdata([t, t])
            lines = [f"t = {t}: halted {f[t]:.1%}", f"undecided {1 - f[t]:.1%} (shaded)"]
            if t >= s_max:
                lines.append(labels["check"])
        else:
            img = bb4_img.copy()
            img[t:] = 4
            image.set_data(img)
            image.set_extent((-0.5, hi4 - lo4 + 0.5, len(img) - 0.5, -0.5))
            title.set_text(f"BB(4) champion alone: step {t - 1}")
            lines = [f"t = {budget}: halted {f[-1]:.1%}", f"undecided {1 - f[-1]:.1%} (shaded)", labels["check"]]
            if t == len(bb4[0]):
                lines.append(labels["bb4"])
        status.set_text("\n".join(lines))

    fig.tight_layout()
    return save_gif(fig, update, frames, path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'states': 3,
    'budget': 100,
    'n_sample': 200000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.turing_halting")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.turing_halting.main")
problem_dir = Path("problems/others/turing_halting")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
